<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='4.%20handling_forms_and_user_input.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 4: Forms</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='6.%20databases_with_flask_sqlalchemy.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 6: Databases →</a>
</div>


# Chapter 5: Sessions and Cookies — Remembering Users

> *"HTTP is stateless — every request is a fresh start. Sessions are how we fake memory."*


## 🎯 The Big Picture

HTTP is a **stateless protocol**. Every request your browser sends is completely independent. The server has no memory of you between requests. After your response is sent, the connection closes and the server forgets you exist.

So how does logging into Gmail mean the *next page* also knows who you are?

The answer is **sessions and cookies** — mechanisms for carrying state across stateless HTTP requests. Without them, you'd have to log in on every single page.

**What you'll learn in this chapter:**
- Why HTTP is stateless and why that creates a problem
- How cookies work at the HTTP level
- How Flask's `session` object provides a secure, dict-like state store
- The `SECRET_KEY` — why it's critical and how to generate a strong one
- Session lifetime, permanence, and the "Remember Me" pattern
- When to use Flask sessions vs raw cookies vs server-side sessions
- Building a complete login/logout flow with session management

## 🧠 Core Concepts: The "Why"

The easiest way to read this section is to keep one plain-language question in view: what is The "Why" actually responsible for? Once that job is clear, the terminology stops feeling arbitrary and the details start to line up.

### The Statelessness Problem

If this section feels dense, pause and identify the data structure or model first. Most of the APIs here make more sense once you know what is being stored, queried, or transformed.

```
Request 1: POST /login  { username: alice, password: secret }
Response 1: 200 OK "Welcome, Alice!"

Request 2: GET /dashboard
Response 2: ??? The server has completely forgotten Alice.
```

Without a way to remember state, users would have to log in on every single page load. That's clearly unusable.
### The Coat Check Analogy

A **session** is exactly like a coat check at a restaurant:

1. **First visit** (first request): You arrive and the attendant gives you a **ticket** (small identifier)
2. **Ticket stored**: Your browser stores the ticket as a **cookie**
3. **Next visit** (next request): You show the ticket → attendant retrieves your coat (session data)
4. **Leaving** (logout): You return the ticket → attendant destroys the record

The **ticket** = the cookie (stored in browser, sent with every request)  
The **coat** = your session data (user ID, preferences, etc.)  
The **coat check system** = Flask's session management

### Three Mechanisms, One Goal

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.

| Mechanism | Stored Where | Signed? | Size Limit | Use Case |
|-----------|-------------|---------|------------|----------|
| Raw Cookie | Browser only | No | ~4KB | Non-sensitive preferences (theme, language) |
| Flask Session | Browser (signed cookie) | ✅ Yes | ~4KB | Login state, user ID, roles |
| Server-side Session | Server (Redis/DB) | N/A | Unlimited | Large data, truly sensitive data |

Flask's default `session` is a **client-side signed session**: the data lives in a cookie, but it's cryptographically signed with your `SECRET_KEY` so users cannot forge or tamper with it.

> ❓ **Why are sessions needed if HTTP already sends cookies?** HTTP is stateless — each request is independent. A session uses a cookie to carry a small identifier (or signed data) so the server can recognise a returning user across multiple requests without requiring a login on every page load.

## ⚡ Syntax & First Usage: The `session` Object

Flask's `session` is a dict-like object available in every route function. Values you store persist across requests for the same user.

In [ ]:
# Flask session — the three core operations
session_patterns = '''
from flask import Flask, session, redirect, url_for

app = Flask(__name__)
app.config["SECRET_KEY"] = "your-secret-key"   # REQUIRED

@app.route("/login", methods=["POST"])
def login():
    # ── Setting values ───────────────────────────────────────────
    session["user_id"]   = 42              # user's database ID
    session["username"]  = "alice"         # display name
    session["is_admin"]  = False           # role
    session["logged_in"] = True            # login flag
    return redirect(url_for("dashboard"))

@app.route("/dashboard")
def dashboard():
    # ── Reading values ───────────────────────────────────────────
    username = session.get("username")     # None if not set (safe)
    user_id  = session.get("user_id")
    role     = session.get("role", "user") # default value

    if not session.get("logged_in"):
        return redirect(url_for("login"))  # not logged in!

    return f"Hello, {username}! Your ID is {user_id}."

@app.route("/logout")
def logout():
    # ── Deleting values ──────────────────────────────────────────
    session.pop("user_id", None)       # remove one key (safe)
    session.pop("username", None)
    session.clear()                    # OR: clear ALL session data at once
    return redirect(url_for("home"))
'''
print(session_patterns)


### The `session` Object Behaves Like a Python Dict

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.



In [ ]:
# Demonstrating session dict operations
# (In a real app these happen inside route functions)
session_sim = {}

print("=== Setting values ===")
session_sim['username'] = 'alice'
session_sim['user_id']  = 42
session_sim['theme']    = 'dark'
print(f"  After setting: {session_sim}")

print()
print("=== Reading values ===")
username = session_sim.get('username')
role     = session_sim.get('role', 'user')      # default value
missing  = session_sim.get('nonexistent')
print(f"  session.get('username')         -> {username!r}")
print(f"  session.get('role', 'user')     -> {role!r}")
print(f"  session.get('nonexistent')      -> {missing!r}")

print()
print("=== Membership check ===")
print(f"  'username' in session           -> {'username' in session_sim}")
print(f"  'password' in session           -> {'password' in session_sim}")

print()
print("=== Deleting one key ===")
session_sim.pop('theme', None)   # safe: no KeyError if key missing
print(f"  After session.pop('theme'):     -> {session_sim}")

print()
print("=== session.clear() for logout ===")
backup = dict(session_sim)
session_sim.clear()
print(f"  Before clear: {backup}")
print(f"  After clear:  {session_sim}")


## 🔬 Deep Dive

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.

### The SECRET_KEY — How Flask Signs Sessions

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.



In [ ]:
# How Flask's signed cookies work
import json, base64, hmac, hashlib

print("=== Step-by-step: How Flask signs a session cookie ===")
print()

SECRET_KEY = b"my-very-secret-key"
session_data = {"user_id": 42, "username": "alice", "is_admin": False}

# Step 1: Serialize to JSON
json_bytes = json.dumps(session_data).encode()
print(f"1. JSON:     {json_bytes}")

# Step 2: Base64 encode
b64_data = base64.b64encode(json_bytes).decode()
print(f"2. Base64:   {b64_data}")

# Step 3: Compute HMAC signature
sig = hmac.new(SECRET_KEY, json_bytes, hashlib.sha256).hexdigest()
print(f"3. HMAC sig: {sig[:32]}... (truncated)")

# Step 4: Cookie = b64_data + "." + timestamp + "." + signature
cookie_value = f"{b64_data}.timestamp.{sig[:16]}"
print(f"4. Cookie:   {cookie_value[:60]}...")

print()
print("=== What happens when a user tries to tamper with their session? ===")
tampered_data = {"user_id": 1, "username": "alice", "is_admin": True}  # changed!
tampered_json = json.dumps(tampered_data).encode()
tampered_sig  = hmac.new(SECRET_KEY, tampered_json, hashlib.sha256).hexdigest()

original_sig_val = hmac.new(SECRET_KEY, json_bytes, hashlib.sha256).hexdigest()
print(f"Original sig:  {original_sig_val[:32]}...")
print(f"Tampered sig:  {tampered_sig[:32]}...")
print(f"Sigs match:    {original_sig_val == tampered_sig}")
print()
print("Flask recomputes the HMAC on every request.")
print("ANY change to the data produces a completely different signature.")
print("Tampered sessions are rejected — user gets logged out.")
print()
print("IMPORTANT: The data is NOT encrypted — it is just Base64 (readable).")
print("Never store passwords, credit cards, or secrets in the session.")
print("Store only the user ID, then look up sensitive data from the DB.")


In [ ]:
import secrets

print("=== Generating a cryptographically strong SECRET_KEY ===")
print()

k1 = secrets.token_hex(32)      # 64 hex chars = 256 bits of entropy
k2 = secrets.token_hex(64)      # 128 hex chars = 512 bits of entropy
k3 = secrets.token_urlsafe(32)  # URL-safe base64

print(f"token_hex(32)      -> {k1!r}")
print(f"  Length: {len(k1)} characters, {len(k1)*4} bits of entropy")
print()
print(f"token_hex(64)      -> {k2!r}")
print(f"  Length: {len(k2)} characters, {len(k2)*4} bits of entropy")
print()
print(f"token_urlsafe(32)  -> {k3!r}")
print(f"  Length: {len(k3)} characters")
print()
print("=== Best practices for SECRET_KEY in production ===")
print()
practices = [
    ("DO",    "Store in environment variable: SECRET_KEY=... in .env file"),
    ("DO",    "Add .env to .gitignore — never commit secrets"),
    ("DO",    "Use os.environ.get('SECRET_KEY') in app.config"),
    ("DO",    "Rotate key if you suspect compromise (invalidates all sessions)"),
    ("DONT",  "Hardcode 'mysecretkey' in source code"),
    ("DONT",  "Use a short or guessable string"),
    ("DONT",  "Commit the real key to version control"),
    ("DONT",  "Use same key in dev and production"),
]
for do_dont, practice in practices:
    print(f"  [{do_dont}]  {practice}")


### Session Lifetime and Permanence — The 'Remember Me' Pattern

Architecture material sounds bigger than it is if you read it only as terminology. Start by identifying the responsibility of each part and the reason those responsibilities are separated; the structure usually becomes much clearer from there.



In [ ]:
from datetime import timedelta

print("=== Two types of sessions ===")
print()
print("1. NON-PERMANENT (default)")
print("   - Session cookie has no Max-Age/Expires")
print("   - Browser deletes cookie when browser tab closes")
print("   - Equivalent to 'don\'t remember me'")
print()
print("2. PERMANENT")
print("   - Cookie has a specific expiry date")
print("   - Survives browser restarts")
print("   - Equivalent to 'remember me for 30 days'")
print()

lifetime_code = '''
from flask import Flask, session, request
from datetime import timedelta

app = Flask(__name__)
app.config["SECRET_KEY"] = "..."
# Default lifetime for PERMANENT sessions:
app.config["PERMANENT_SESSION_LIFETIME"] = timedelta(days=30)

@app.route("/login", methods=["POST"])
def login():
    remember_me = request.form.get("remember_me") == "on"

    # After verifying credentials...
    session["user_id"]  = 42
    session["username"] = "alice"

    if remember_me:
        session.permanent = True   # use PERMANENT_SESSION_LIFETIME
    # If not remember_me: session.permanent stays False (browser-close expiry)

    return redirect(url_for("dashboard"))
'''
print(lifetime_code)

print()
print("Common PERMANENT_SESSION_LIFETIME values:")
durations = [
    ("15 minutes",  timedelta(minutes=15)),
    ("1 hour",      timedelta(hours=1)),
    ("1 day",       timedelta(days=1)),
    ("1 week",      timedelta(weeks=1)),
    ("30 days",     timedelta(days=30)),
]
for name, td in durations:
    secs = int(td.total_seconds())
    print(f"  timedelta({name:<15}) = {secs:>8} seconds")


### Raw Cookies — The Lower-Level Mechanism

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.



In [ ]:
raw_cookies = '''
from flask import Flask, request, make_response, redirect, url_for

# When to use raw cookies instead of session:
#   - Non-sensitive user preferences (theme, language, items per page)
#   - Data that doesn't need to be tied to login state
#   - Long-lived preferences that should survive logout

@app.route("/set-theme/<theme>")
def set_theme(theme):
    response = make_response(redirect(url_for("home")))
    response.set_cookie(
        key="theme",             # cookie name
        value=theme,             # cookie value ("dark" or "light")
        max_age=365*24*60*60,    # 1 year in seconds
        httponly=True,           # JS cannot read this (prevents XSS theft)
        samesite="Lax",          # CSRF protection at cookie level
        secure=True,             # only send over HTTPS (use in production)
    )
    return response

@app.route("/")
def home():
    theme = request.cookies.get("theme", "light")   # default to "light"
    return f"Current theme: {theme}"

@app.route("/reset-theme")
def reset_theme():
    response = make_response(redirect(url_for("home")))
    response.delete_cookie("theme")   # sets Max-Age=0 (browser deletes it)
    return response
'''
print(raw_cookies)
print()
print("Cookie security flags:")
flags = [
    ("HttpOnly",  "Cookie cannot be read by JavaScript. Prevents XSS theft."),
    ("Secure",    "Cookie only sent over HTTPS. Use in production always."),
    ("SameSite",  "Lax: safe default. Strict: never cross-site. None: always (needs Secure)."),
    ("Max-Age",   "Seconds until expiry. Overrides Expires. 0 = delete immediately."),
    ("Domain",    "Which domains receive this cookie. Default: exact current domain only."),
    ("Path",      "URL path scope. Default: / (all paths on domain)."),
]
for flag, desc in flags:
    print(f"  {flag:<12}  {desc}")


### ⚖️ Sessions vs Raw Cookies vs Server-Side Sessions

Sections like this become clearer when you separate capabilities from tradeoffs. As you read, ask what each option is optimized for and what complexity you accept in exchange.



In [ ]:
lines = [
    "+-----------------------+-------------------+--------------------+------------------------+",
    "| Feature               | Raw Cookie        | Flask Session      | Server-Side Session    |",
    "+-----------------------+-------------------+--------------------+------------------------+",
    "| Storage location      | Browser           | Browser (signed)   | Server (Redis/DB)      |",
    "| Tamper protection     | None              | HMAC signature     | N/A (ID only in cookie)|",
    "| Data visibility       | Plain text        | Base64 readable    | Not in cookie          |",
    "| Size limit            | ~4KB              | ~4KB               | Unlimited              |",
    "| Can store secrets?    | No                | No (visible)       | Yes                    |",
    "| Invalidate server-side| No                | No                 | Yes (true logout)      |",
    "| Scales horizontally?  | Yes               | Yes                | Needs shared Redis     |",
    "| Setup complexity      | Minimal           | Just SECRET_KEY    | Needs Redis/DB         |",
    "| Use case              | Preferences       | Login state        | Sensitive/large data   |",
    "+-----------------------+-------------------+--------------------+------------------------+",
    "",
    "Rule of thumb:",
    "  Non-sensitive prefs (theme, language)  ->  raw cookie",
    "  Login state, user ID, roles            ->  Flask session (default)",
    "  Sensitive data, large data             ->  Flask-Session with Redis",
]
for line in lines:
    print(line)


### Complete Login/Logout Flow with Sessions

Whenever a process has several stages, beginners benefit from tracing the handoff between them. Follow the order carefully and notice what information each step receives, transforms, and passes on.



In [ ]:
complete_app = '''
from flask import Flask, session, redirect, url_for, request, flash, render_template
from functools import wraps

app = Flask(__name__)
app.config["SECRET_KEY"] = "strong-secret-key"
app.config["PERMANENT_SESSION_LIFETIME"] = timedelta(days=30)

# ── @login_required decorator ────────────────────────────────────
# Apply to any route that requires login
def login_required(f):
    @wraps(f)
    def decorated_function(*args, **kwargs):
        if "user_id" not in session:
            flash("Please log in to access this page.", "warning")
            # Preserve the URL they tried to visit
            return redirect(url_for("login", next=request.url))
        return f(*args, **kwargs)
    return decorated_function

# ── Login ─────────────────────────────────────────────────────────
@app.route("/login", methods=["GET", "POST"])
def login():
    if "user_id" in session:
        return redirect(url_for("dashboard"))   # already logged in

    if request.method == "POST":
        username   = request.form.get("username", "")
        password   = request.form.get("password", "")
        remember   = request.form.get("remember_me") == "on"

        # Real app: query database + verify password hash
        if username == "alice" and password == "password123":
            if remember:
                session.permanent = True
            session["user_id"]  = 42
            session["username"] = username

            next_page = request.args.get("next")  # where they were going
            flash(f"Welcome back, {username}!", "success")
            return redirect(next_page or url_for("dashboard"))
        else:
            flash("Invalid username or password.", "danger")

    return render_template("login.html")

# ── Protected routes ──────────────────────────────────────────────
@app.route("/dashboard")
@login_required
def dashboard():
    return f"Dashboard for {session.get('username')} (ID: {session.get('user_id')})"

@app.route("/admin")
@login_required
def admin_panel():
    if not session.get("is_admin"):
        flash("Admin access required.", "danger")
        return redirect(url_for("dashboard"))
    return "Admin Panel"

# ── Logout ────────────────────────────────────────────────────────
@app.route("/logout")
def logout():
    username = session.get("username", "User")
    session.clear()    # Remove all session keys at once
    flash(f"Goodbye, {username}! You have been logged out.", "info")
    return redirect(url_for("login"))
'''
print(complete_app)


## 🧪 What If? Experimentation

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

### What If 1: You Don't Set SECRET_KEY?

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.



In [ ]:
print('''
SCENARIO: You start using sessions without setting SECRET_KEY

What happens:
  1. App starts successfully (no error at startup)
  2. User visits a route that reads or writes session
  3. Flask tries to sign/verify the session cookie
  4. No SECRET_KEY -> RuntimeError is raised!

Error message:
  RuntimeError: The session is unavailable because no secret key
  was set. Set the secret_key on the application to something
  unique and secret.

The error appears at RUNTIME (first session access), not at import.
This means it can be easy to miss during development if you
test without logging in.

Prevention:
  # In app.py or config.py:
  import os
  app.config["SECRET_KEY"] = os.environ.get("SECRET_KEY", "dev-only-fallback")

  # For development only (weak key):
  app.config["SECRET_KEY"] = "dev-not-for-production"

  # Generate a strong key (run once, save to .env):
  # python -c "import secrets; print(secrets.token_hex(32))"
''')


### What If 2: You Store Too Much Data in Session?

If this section feels dense, pause and identify the data structure or model first. Most of the APIs here make more sense once you know what is being stored, queried, or transformed.



In [ ]:
import json, base64

print("=== Cookie size limit demonstration ===")
print()

# Small session (typical — just the essentials)
small = {"user_id": 42, "username": "alice", "is_admin": False}
small_json = json.dumps(small)
small_b64  = base64.b64encode(small_json.encode())
print(f"SMALL session: {small}")
print(f"  JSON size:   {len(small_json)} bytes")
print(f"  B64 size:    {len(small_b64)} bytes")
print(f"  Safe: {'YES' if len(small_b64) < 3500 else 'NO'}")

print()
# Large session (antipattern — storing too much)
large = {
    "user_id": 42,
    "username": "alice",
    "full_profile": {
        "bio": "A" * 1500,
        "interests": ["coding"] * 100,
        "recent_posts": list(range(50)),
    }
}
large_json = json.dumps(large)
large_b64  = base64.b64encode(large_json.encode())
print(f"LARGE session JSON: {len(large_json)} bytes")
print(f"  B64 size:  {len(large_b64)} bytes")
print(f"  Over 4KB:  {len(large_b64) > 4096}")

print()
print("What happens when cookie exceeds ~4KB:")
print("  - Browser SILENTLY drops the cookie (no error!)")
print("  - Session data is LOST")
print("  - User appears to be magically logged out")
print("  - Extremely confusing bug to diagnose")
print()
print("Best practices:")
print("  - Store ONLY the user ID in session")
print("  - Load full user data from DB in each request")
print("  - Use g.current_user = User.query.get(session['user_id'])")
print("  - For large session data: use Flask-Session with Redis")


### What If 3: A User Manually Edits Their Session Cookie?

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.



In [ ]:
import hmac, hashlib, json, base64

SECRET_KEY = b"super-secret"

print("=== User tries to forge a session by editing cookie ===")
print()

# Legitimate session: user_id=42, is_admin=False
legit_data = json.dumps({"user_id": 42, "is_admin": False}).encode()
legit_b64  = base64.b64encode(legit_data).decode()
legit_sig  = hmac.new(SECRET_KEY, legit_data, hashlib.sha256).hexdigest()
print(f"Legitimate session:")
print(f"  data: {json.loads(legit_data)}")
print(f"  b64:  {legit_b64}")
print(f"  sig:  {legit_sig[:32]}...")

print()
# Attacker tries to change is_admin to True
forged_data = json.dumps({"user_id": 42, "is_admin": True}).encode()
forged_b64  = base64.b64encode(forged_data).decode()
# Attacker doesn't know SECRET_KEY so can't compute valid sig
# They just send the tampered data with the old signature
print(f"Forged session (is_admin=True):")
print(f"  data: {json.loads(forged_data)}")
print(f"  b64:  {forged_b64}")

print()
# Flask verifies by recomputing HMAC
expected_sig = hmac.new(SECRET_KEY, forged_data, hashlib.sha256).hexdigest()
print(f"Expected sig: {expected_sig[:32]}...")
print(f"Provided sig: {legit_sig[:32]}...")
print(f"Sigs match:   {expected_sig == legit_sig}")
print()
print("Result: REJECTED. User gets logged out. Attack fails.")
print()
print("Why SECRET_KEY must stay secret:")
print("  If attacker knows SECRET_KEY, they can compute a valid HMAC")
print("  for ANY forged session -> complete account takeover.")
print("  Never log, print, or expose SECRET_KEY.")


## 🚀 Real-World Project Link

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

Our **Blog** uses `session['user_id']` to maintain login state across all pages. A `@login_required` decorator checks for this key and redirects unauthenticated users to the login page. The login form's "Remember Me" checkbox sets `session.permanent = True` with a 30-day lifetime. After every action (post created, comment deleted), a flash message stored in the session appears on the next page then disappears permanently.

## 📋 Chapter Summary & Cheat Sheet

In [ ]:
lines = [
    "# ============================================================",
    "#     CHAPTER 5 CHEAT SHEET — SESSIONS & COOKIES",
    "# ============================================================",
    "",
    "# REQUIRED SETUP",
    'app.config["SECRET_KEY"] = os.environ.get("SECRET_KEY")',
    "# Generate key: python -c "import secrets; print(secrets.token_hex(32))"",
    "",
    "# SESSION OPERATIONS",
    'session["user_id"]  = 42          # set value',
    'session["username"] = "alice"',
    'session.get("user_id")            # safe read (None if missing)',
    'session.get("role", "user")        # with default',
    '"user_id" in session               # check existence',
    'session.pop("user_id", None)       # remove one key (safe)',
    "session.clear()                    # remove ALL keys (use for logout)",
    "",
    "# PERMANENT SESSION (Remember Me)",
    "from datetime import timedelta",
    'app.config["PERMANENT_SESSION_LIFETIME"] = timedelta(days=30)',
    "session.permanent = True           # mark this session as long-lived",
    "",
    "# RAW COOKIES",
    "response = make_response(redirect(url_for('home')))",
    'response.set_cookie("theme", "dark", max_age=365*24*60*60, httponly=True)',
    'theme = request.cookies.get("theme", "light")',
    'response.delete_cookie("theme")',
    "",
    "# LOGIN REQUIRED DECORATOR",
    "from functools import wraps",
    "def login_required(f):",
    "    @wraps(f)",
    "    def decorated(*args, **kwargs):",
    '        if "user_id" not in session:',
    '            flash("Please log in.", "warning")',
    '            return redirect(url_for("login"))',
    "        return f(*args, **kwargs)",
    "    return decorated",
    "",
    "# COMPLETE LOGOUT",
    "session.clear()",
    'flash("Logged out.", "info")',
    'return redirect(url_for("login"))',
]
for line in lines:
    print(line)


### The `g` Object — Per-Request Context

The key to this section is sequence. If you can explain what happens first, what happens next, and where control moves after that, the moving parts here become much easier to reason about.

While `session` persists across requests, Flask's `g` object is a **request-scoped storage** for data that should be available throughout the current request only.

In [ ]:
# Flask's g object — per-request storage
g_example = '''
from flask import Flask, g, session, request

app = Flask(__name__)
app.config["SECRET_KEY"] = "secret"

# Common pattern: load the current user ONCE per request
# Store it on g so every function in this request can access it

@app.before_request
def load_current_user():
    user_id = session.get("user_id")
    if user_id:
        g.current_user = User.query.get(user_id)
    else:
        g.current_user = None

# Now in any route, template, or helper:
@app.route("/profile")
def profile():
    if g.current_user is None:
        return redirect(url_for("login"))
    return f"Hello, {g.current_user.username}"

# Also available in Jinja2 templates:
# {{ g.current_user.username if g.current_user else "Guest" }}

# g vs session:
#   g        = lives for ONE request (gone after response is sent)
#   session  = lives across MULTIPLE requests (stored in cookie)
#   Use g to avoid repeated DB queries within a single request
#   Use session to remember state across requests
'''
print(g_example)


### Session-Based Flash Messages Deep Dive

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.



In [ ]:
# How flash messages are stored in the session
flash_internals = '''
from flask import session, get_flashed_messages

# Under the hood, flash() does:
#   session["_flashes"] = session.get("_flashes", []) + [(category, message)]

# And get_flashed_messages() does:
#   messages = session.pop("_flashes", [])  # removes from session!
#   return messages

# This means:
#   1. Flash messages ARE session data (count toward the 4KB limit)
#   2. They are consumed exactly once
#   3. If get_flashed_messages() is NOT called in the template,
#      messages accumulate and are shown on the NEXT page

# Practical example: notification system
@app.route("/follow/<int:user_id>", methods=["POST"])
def follow_user(user_id):
    # ... create follow relationship ...
    flash(f"You are now following {followed_user.username}!", "success")
    return redirect(url_for("profile", username=followed_user.username))

# In base.html — shows messages from ANY previous action
# {% with messages = get_flashed_messages(with_categories=true) %}
#   {% if messages %}
#     <div id="notifications" style="position:fixed; top:1em; right:1em;">
#       {% for cat, msg in messages %}
#         <div class="toast toast-{{ cat }}" data-autohide="true">
#           {{ msg }}
#         </div>
#       {% endfor %}
#     </div>
#   {% endif %}
# {% endwith %}
'''
print(flash_internals)


### Implementing 'Remember Me' with Flask-Login (Preview)

The `session.permanent` pattern we learned works well manually, but **Flask-Login** is the standard library for session-based authentication in production Flask apps.

In [ ]:
flask_login_preview = '''
# pip install flask-login

from flask_login import LoginManager, UserMixin, login_user, logout_user
from flask_login import login_required, current_user

login_manager = LoginManager(app)
login_manager.login_view = "login"       # redirect here if not logged in
login_manager.login_message = "Please log in to access this page."
login_manager.login_message_category = "warning"

# Your User model mixes in UserMixin (provides is_authenticated, is_active, etc.)
class User(db.Model, UserMixin):
    id       = db.Column(db.Integer, primary_key=True)
    username = db.Column(db.String(80), unique=True, nullable=False)

    # UserMixin provides these automatically:
    #   is_authenticated -> True for logged-in users
    #   is_active        -> True (override for banned/deactivated accounts)
    #   is_anonymous     -> False (AnonymousUser is True)
    #   get_id()         -> str(self.id) used by session

# Required: tell Flask-Login how to load a user from the session
@login_manager.user_loader
def load_user(user_id):
    return User.query.get(int(user_id))

# Login
@app.route("/login", methods=["GET", "POST"])
def login():
    if current_user.is_authenticated:
        return redirect(url_for("dashboard"))
    form = LoginForm()
    if form.validate_on_submit():
        user = User.query.filter_by(username=form.username.data).first()
        if user and check_password_hash(user.password_hash, form.password.data):
            login_user(user, remember=form.remember_me.data)  # one line!
            return redirect(url_for("dashboard"))
        flash("Invalid credentials.", "danger")
    return render_template("login.html", form=form)

# Logout
@app.route("/logout")
@login_required
def logout():
    logout_user()
    flash("You have been logged out.", "info")
    return redirect(url_for("login"))

# Protected route — just add decorator
@app.route("/dashboard")
@login_required
def dashboard():
    return f"Hello, {current_user.username}!"   # current_user is the User object!
'''
print(flask_login_preview)


### Session Security Hardening

Security topics become much easier to follow when you separate the threat from the defense. As you read, keep asking what can go wrong, what protection addresses it, and what assumption that protection depends on.



In [ ]:
# Additional session security configurations
security_config = '''
# Harden Flask session cookies for production:

app.config["SECRET_KEY"]          = os.environ["SECRET_KEY"]
app.config["SESSION_COOKIE_HTTPONLY"]  = True    # JS cannot read session cookie
app.config["SESSION_COOKIE_SECURE"]    = True    # HTTPS only (production!)
app.config["SESSION_COOKIE_SAMESITE"]  = "Lax"  # CSRF protection
app.config["SESSION_COOKIE_NAME"]      = "session"  # default
app.config["SESSION_COOKIE_DOMAIN"]    = None    # current domain only

# For development (HTTP), set SECURE=False:
if app.debug:
    app.config["SESSION_COOKIE_SECURE"] = False

# Summary of all session cookie flags:
# HttpOnly: True  -> prevents XSS scripts from reading the cookie
# Secure:   True  -> only sent over HTTPS (prevents interception)
# SameSite: Lax   -> prevents CSRF (sent on same-site navigations)
#           Strict -> never on cross-site requests (too strict for most apps)
#           None   -> always (requires Secure=True)

# What happens without these protections:
# No HttpOnly -> malicious JS can read session cookie (XSS -> session hijack)
# No Secure   -> cookie sent over HTTP -> intercepted (man-in-middle)
# No SameSite -> CSRF attacks possible (same as no CSRF protection)
'''
print(security_config)


## 💪 Practice Prompts

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

**Challenge 1 — Shopping Cart:**
Build a `/add-to-cart/<int:product_id>` route that appends the product ID to `session['cart']` (a list). Use `session.setdefault('cart', [])` to initialize it on first access. Build `/cart` to display the list and `/clear-cart` to empty it. Test that the cart persists across page loads.

**Challenge 2 — Visit Counter with Timestamp:**
Create a `/counter` route that tracks how many times the current user has visited. Store `session['visit_count']` (integer) and `session['first_visit']` (ISO timestamp string from `datetime.utcnow().isoformat()`). Display both on the page. Add a "Reset" button (`/reset-counter`) that clears the counter but NOT the first_visit timestamp.

**Challenge 3 — Theme Preference with Cookie:**
Build a `/toggle-theme` route that switches between 'light' and 'dark' using a **raw cookie** (not a session). Cookie should last 365 days with `httponly=True` and `samesite='Lax'`. The template reads the cookie and applies a CSS background color to `<body>`. Explain in a comment why a raw cookie (not session) is appropriate for this use case.


<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='4.%20handling_forms_and_user_input.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 4: Forms</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='6.%20databases_with_flask_sqlalchemy.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 6: Databases →</a>
</div>

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='4. handling_forms_and_user_input.ipynb' style='font-weight:bold; font-size:1.05em;'>&larr; Previous</a>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <a href='6. databases_with_flask_sqlalchemy.ipynb' style='font-weight:bold; font-size:1.05em;'>Next &rarr;</a>
</div>
